# 01 Select an archetype

This notebook loads the Swedish LoD2 archetype catalogue, shows what it covers and
selects one archetype for a building. It then answers the questions a modeller asks
first: what are the U-values, what is the window, how leaky is the envelope, and where
do the values come from?

**Prerequisites:** Python 3.11+ and `epsm_data` installed (`python -m pip install -e .`).
The catalogue ships with the package, so the notebook runs offline. U-values are in
W/(m² K); infiltration rates state their own reference basis.

Load the bundled catalogue. Printing it gives a one-line description.

In [ ]:
from epsm_data import Catalogue

catalogue = Catalogue.load()
catalogue

Catalogue(id='epsm-lod2', families=28, configurations=76)

`summary()` counts the archetypes, their configurations and any missing values.

In [ ]:
summary = catalogue.summary()
for key in ("archetype_count", "configuration_count", "slot_count", "incomplete_slots"):
    print(f"{key:<20} {summary[key]}")

archetype_count      28
configuration_count  76
slot_count           228
incomplete_slots     0


The catalogue covers four building types, seven construction eras and three renovation
states. Each configuration is offered in three climate zones.

In [ ]:
dimensions = catalogue.dimensions
print("Types: ", ", ".join(dimensions["categories"]))
for era, (first, last) in dimensions["eras"].items():
    print(f"{era}: {first}-{last or 'today'}")
print("States:", ", ".join(dimensions["states"]))
print("Zones: ", ", ".join(dimensions["zones"]))

Types:  single_family, multi_family, office, school
SE.01: 1-1960
SE.02: 1961-1975
SE.03: 1976-1985
SE.04: 1986-1995
SE.05: 1996-2005
SE.06: 2006-2021
SE.07: 2022-today
States: existing, standard, advanced
Zones:  SE.Zone1, SE.Zone2, SE.Zone3


Eras SE.01 to SE.06 have an existing, a standard-renovated and an advanced-renovated
state. SE.07 (2022 onwards) is new construction and has a single configuration, selected
with `state=None`.

In [ ]:
for archetype_id in ("mf_se02", "mf_se07"):
    states = catalogue.renovation_states(archetype_id)
    print(archetype_id, states or "single configuration (state=None)")

mf_se02 ('existing', 'standard', 'advanced')
mf_se07 single configuration (state=None)


The fabric is the same in every climate zone; the zone only travels with the selection.
`climate_zone_for_county` finds the zone of a Swedish county.

In [ ]:
from epsm_data import climate_zone_for_county

zone = climate_zone_for_county("Västra Götalands län")
slot = catalogue.get_slot(f"multi_family:SE.02:{zone}:existing")
print(zone, slot.archetype_id, "complete" if slot.complete else slot.missing_values)

SE.Zone3 mf_se02 complete


Select an archetype for a multi-family building built in 1965, as it stands today.

In [ ]:
print("Era:", catalogue.era_for_year(1965))
archetype = catalogue.for_year("multi_family", 1965, state="existing")
print(archetype.display_name, "|", archetype.construction_year)

Era: SE.02
Multi-family SE.02 · existing | 1965


If you already know the archetype ID, `get` returns the same definition.

In [ ]:
same = catalogue.get("mf_se02", state="existing")
assert same.envelope_u_values == archetype.envelope_u_values

Nominal U-value of each construction role. Ground and basement values are the material
layers alone, not an effective ground U-value; the simulation adds the soil.

In [ ]:
for role, element in archetype.construction_set.elements.items():
    print(f"{role.value:<25} U = {element.u_value:5.2f}")

BASEMENT_WALL             U =  3.17
EXT_WALL                  U =  0.42
GROUND_FLOOR              U =  0.63
INTERMEDIATE_FLOOR        U =  1.77
INT_WALL_NONLOADBEARING   U =  0.60
TOP_FLOOR                 U =  0.20
WINDOW                    U =  2.22


The window is a whole-window U-value with a solar heat gain coefficient (g-value).

In [ ]:
window = archetype.window
print(f"U = {window.u_value} W/(m² K), g = {window.shgc}, visible transmittance = {window.vt}")

U = 2.22 W/(m² K), g = 0.56, visible transmittance = 0.525


The envelope performance holds infiltration and the thermal-bridge assumption.
Residential archetypes leave the thermal bridge as unknown (`None`).

In [ ]:
envelope = archetype.envelope_performance
print(envelope)
print("Infiltration:", envelope.infiltration_l_s_m2, "l/(s m²) per", envelope.infiltration_reference_basis)
print("Thermal-bridge surcharge:", envelope.thermal_bridge_delta_u_w_m2k)

EnvelopePerformance(l_s_m2=0.2, basis='Atemp', bridges=unknown, sources=0)
Infiltration: 0.2 l/(s m²) per Atemp
Thermal-bridge surcharge: None


Office states carry a thermal-bridge surcharge on the whole envelope U-value, with
its sources and the reasoning behind it.

In [ ]:
office = catalogue.for_year("office", 1978, state="existing")
envelope = office.envelope_performance
print(office.display_name, "| ΔU =", envelope.thermal_bridge_delta_u_w_m2k, "W/(m² K)")
for reference in envelope.source_references:
    print(" source:", reference)
for note in envelope.notes:
    print(" note:  ", note)

Office SE.03 · existing | ΔU = 0.1 W/(m² K)
 source: TABULA.xlsm Calc.Building.Set delta_U_ThermalBridging, Swedish multi-family type of the same era and state
 source: epsm_studies validation/catalogue/calibration_plan.md, Fabric revision 1: accepted for offices on the calibration and hold-out samples
 note:   TABULA has no non-residential types; offices take the multi-family surcharge of their era and state.
 note:   SE.06 and SE.07 take 0.05 W/(m2 K), the lowest non-zero Swedish TABULA value; TABULA has no Swedish type after 2005.


Construction sources sit on the materials: each layer names its Boverket climate
database resource. The catalogue records which source releases it was built from.

In [ ]:
from epsm_data import BuildingElement

print("Source releases:", dict(catalogue.source_versions))
wall = archetype.construction_set.elements[BuildingElement.EXT_WALL]
for component in wall.components:
    for slot in component.layers:
        for share in slot.layer.materials:
            print(f"  {share.material.name:<50} Boverket {share.material.boverket_id}")

Source releases: {'boverket': '02.07.000', 'epd-ies-0005222': '003', 'epd-ies-0013025': '003', 'epsm-custom': '0.9.0'}
  Clay brick, lambda 0.60 W/(m K), 1650 kg/m³        Boverket 6000000082
  Mortar                                             Boverket 6000000092
  Gypsum wind sheathing                              Boverket 6000000015
  Stone wool batts                                   Boverket 6000000123
  Softwood timber, 500 kg/m³                         Boverket 6000000007
  Gypsum plasterboard, Gyproc GN, 720 kg/m³          Boverket 6000000020
  Gypsum plasterboard, Gyproc GN, 720 kg/m³          Boverket 6000000020


Any modelling notes on the selection, such as stock with no carbon factor, are kept on the archetype.

In [ ]:
import textwrap

for note in archetype.notes:
    print("-", textwrap.shorten(note, 110))

- Retained existing stock without quantified carbon: material:622fee5beaa8a10c. Hardboard is a different [...]
